**Dependency Installation**

In [1]:
!pip install -U langgraph langchain-groq pydantic | tail -n 1

**Initialize groq api key**

In [4]:
import os
from google.colab import userdata
# 1. Set Groq API Key
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

**Define State & Output Schemas**

In [5]:
from typing import TypedDict, List
from pydantic import BaseModel, Field

# --- Pydantic Schemas for Structured LLM Outputs ---
class ReviewIssue(BaseModel):
    category: str = Field(description="e.g., Security, Performance, Style")
    severity: str = Field(description="High, Medium, Low")
    description: str = Field(description="Detailed explanation of the issue")
    line_hint: str = Field(description="Relevant code snippet or line reference")

class AgentReview(BaseModel):
    agent_name: str
    issues: List[ReviewIssue]
    summary: str

# --- LangGraph Shared State ---
class CodeReviewState(TypedDict):
    code: str
    language: str
    security_review: str
    performance_review: str
    final_refactored_code: str
    review_summary: str

**Initialize the Groq Model**

In [10]:
import os
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1,
)

**Define Agent Nodes**

In [11]:
from langchain_core.prompts import ChatPromptTemplate

# 1. Security & Quality Agent
def security_agent(state: CodeReviewState) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an Expert Application Security Engineer. Analyze the code for security vulnerabilities, hardcoded secrets, injection threats, and memory leaks."),
        ("human", "Language: {language}\nCode:\n```{language}\n{code}\n```")
    ])
    chain = prompt | llm.with_structured_output(AgentReview)
    result: AgentReview = chain.invoke({"language": state["language"], "code": state["code"]})

    # Format review string
    issues_str = "\n".join([f"- [{i.severity}] {i.category}: {i.description} ({i.line_hint})" for i in result.issues])
    review_output = f"Summary: {result.summary}\nIssues:\n{issues_str}"

    return {"security_review": review_output}

# 2. Performance & Clean Code Agent
def performance_agent(state: CodeReviewState) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a Principal Software Architect. Analyze the code for time/space complexity, redundant operations, poor naming, and readability improvements."),
        ("human", "Language: {language}\nCode:\n```{language}\n{code}\n```")
    ])
    chain = prompt | llm.with_structured_output(AgentReview)
    result: AgentReview = chain.invoke({"language": state["language"], "code": state["code"]})

    issues_str = "\n".join([f"- [{i.severity}] {i.category}: {i.description} ({i.line_hint})" for i in result.issues])
    review_output = f"Summary: {result.summary}\nIssues:\n{issues_str}"

    return {"performance_review": review_output}

# 3. Lead Synthesizer Agent
def lead_reviewer_agent(state: CodeReviewState) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are the Lead Code Reviewer.
          Synthesize the Security and Performance reports into a final refactored code version.
          Fix all critical issues while keeping the code idiomatic and clean.

          Output format:
          ## Summary of Changes
          <bullet points>

          ## Refactored Code
          ```{language}
          <code here>

          """),
          ("human", """Original Code ({language}):

          {code}
          Security Audit:
          {security_review}

          Performance Audit:
          {performance_review}
          """)
        ])
    chain = prompt | llm
    response = chain.invoke({
      "language": state["language"],
      "code": state["code"],
      "security_review": state["security_review"],
      "performance_review": state["performance_review"]
    })
    content = response.content
    return ({
        "final_refactored_code": content,
        "review_summary": "Review and refactoring complete."
    })

**Construct and Compile the LangGraph**

In [12]:
### Step 5: Construct and Compile the LangGraph
from langgraph.graph import StateGraph, START, END

# Initialize Graph
builder = StateGraph(CodeReviewState)

# Add Nodes
builder.add_node("security_agent", security_agent)
builder.add_node("performance_agent", performance_agent)
builder.add_node("lead_reviewer", lead_reviewer_agent)

# Define Edges (Parallel Execution -> Fan-in)
builder.add_edge(START, "security_agent")
builder.add_edge(START, "performance_agent")
builder.add_edge("security_agent", "lead_reviewer")
builder.add_edge("performance_agent", "lead_reviewer")
builder.add_edge("lead_reviewer", END)

# Compile
app = builder.compile()

**Execute the Workflow**

In [13]:
sample_code = """
import os
import sqlite3

def get_user_data(user_id):
    # Potential SQL injection and unclosed resource
    conn = sqlite3.connect('database.db')
    cursor = conn.cursor()
    query = "SELECT * FROM users WHERE id = '" + str(user_id) + "'"
    cursor.execute(query)
    data = cursor.fetchall()

    # Inefficient loop
    results = []
    for item in data:
        if item not in results:
            results.append(item)

    return results
"""

inputs = {
    "code": sample_code,
    "language": "python"
}

# Run the Graph
final_state = app.invoke(inputs)

print("=== SECURITY REVIEW ===")
print(final_state["security_review"])

print("\n=== PERFORMANCE REVIEW ===")
print(final_state["performance_review"])

print("\n=== FINAL REFACTORED OUTPUT ===")
print(final_state["final_refactored_code"])

=== SECURITY REVIEW ===
Summary: The function suffers from a critical SQL injection vulnerability due to string concatenation of user input, and it leaks database resources by not closing connections. Additional concerns include lack of input validation, missing error handling, and inefficient duplicate removal. Refactoring to use parameterized queries, proper resource cleanup (e.g., context managers), input validation, and optimized deduplication will mitigate these issues.
Issues:
- [High] Security: The code constructs an SQL query by concatenating user-supplied input (`user_id`) directly into the query string, which enables SQL injection attacks. An attacker could supply a crafted `user_id` value to manipulate the query, potentially extracting or modifying data. (query = "SELECT * FROM users WHERE id = '" + str(user_id) + "'")
- [Medium] Resource Management: The SQLite connection and cursor are never explicitly closed, which can lead to resource leaks, especially under high load or 

**LLM-as-a-Judge Evaluation (Quality & Relevance)**

- Security Agent Accuracy: Did the security agent catch true positives without hallucinating fake vulnerabilities?

- Performance Agent Accuracy: Did it identify real performance bottlenecks?

- Synthesis Quality (Lead Reviewer): Did the lead reviewer incorporate feedback from both agents without dropping critical fixes or introducing new bugs?

- Code Soundness / Functional Equivalence: Does the refactored code preserve the original business logic?

In [14]:
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

class EvalScore(BaseModel):
    functional_equivalence: int = Field(description="1-5 rating: Does the code retain original logic?")
    security_fix_score: int = Field(description="1-5 rating: Were security risks successfully resolved?")
    synthesis_score: int = Field(description="1-5 rating: Did the lead reviewer address both agents' feedback?")
    reasoning: str = Field(description="Explanation of the assigned ratings")

eval_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.0)

eval_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI Evaluator testing a code refactoring system. Grade the transformation objectively."),
    ("human", """Original Code:
{original_code}

Security Feedback:
{security_review}

Performance Feedback:
{performance_review}

Final Refactored Output:
{refactored_output}
""")
])

eval_chain = eval_prompt | eval_llm.with_structured_output(EvalScore)

# Run evaluation on final_state
evaluation = eval_chain.invoke({
    "original_code": final_state["code"],
    "security_review": final_state["security_review"],
    "performance_review": final_state["performance_review"],
    "refactored_output": final_state["final_refactored_code"]
})

print(f"Scores: {evaluation.model_dump_json(indent=2)}")

Scores: {
  "functional_equivalence": 4,
  "security_fix_score": 5,
  "synthesis_score": 5,
  "reasoning": "The refactoring incorporates both the security and performance feedback, adding proper resource management, deduplication optimization, documentation, type hints, and custom error handling, fully addressing the reviewers' concerns."
}
